## **Imports and Configs**

In [10]:
import json
import math
import pandas as pd

In [11]:
JSON_FILE = "/content/scifact_retrieval_eval_results.json"

K_VALUES = [3, 5, 7, 10, 20, 50]

METHODS = {
    "BM25": "bm25_only",
    "BM25 + query expansion": "expansion_only",
    "BM25 + reranking": "reranking_only",
    "BM25 + query expansion + reranking": "expansion_plus_reranking",
}

## **Load Results**

In [12]:
with open(JSON_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Number of queries: {len(data)}")

Number of queries: 300


## **Helper Functions**

In [13]:
def get_doc_id(item):
    """
    Extract document ID from a ranked result.

    Your JSON uses:
      bm25_only / expansion_only:
          {"doc_id": ..., "score": ...}

      reranking_only / expansion_plus_reranking:
          {"doc_id": ..., "fusion_score": ..., ...}

    All of them have doc_id, so this function keeps things
    robust in case the structure changes slightly.
    """
    return item["doc_id"]


def recall_at_k(ranked_results, gold_doc_ids, k):
    """
    Recall@k = relevant documents retrieved in top-k /
               total number of relevant documents.
    """

    if not gold_doc_ids:
        return 0.0

    gold_set = set(gold_doc_ids)

    retrieved = {
        get_doc_id(item)
        for item in ranked_results[:k]
    }

    relevant_retrieved = len(retrieved & gold_set)

    return relevant_retrieved / len(gold_set)


def reciprocal_rank_at_k(ranked_results, gold_doc_ids, k):
    """
    MRR@k for a single query.

    Returns reciprocal rank of the FIRST relevant document
    in the top-k. Returns 0 if no relevant document occurs.
    """

    gold_set = set(gold_doc_ids)

    for rank, item in enumerate(ranked_results[:k], start=1):
        if get_doc_id(item) in gold_set:
            return 1.0 / rank

    return 0.0


def dcg_at_k(ranked_results, gold_doc_ids, k):
    """
    DCG@k with binary relevance.

    relevance = 1 if doc is a gold document, otherwise 0.
    """

    gold_set = set(gold_doc_ids)

    dcg = 0.0

    for rank, item in enumerate(ranked_results[:k], start=1):
        relevance = 1 if get_doc_id(item) in gold_set else 0

        if relevance:
            dcg += relevance / math.log2(rank + 1)

    return dcg


def ndcg_at_k(ranked_results, gold_doc_ids, k):
    """
    NDCG@k with binary relevance.
    """

    if not gold_doc_ids:
        return 0.0

    # Actual DCG
    dcg = dcg_at_k(ranked_results, gold_doc_ids, k)

    # Ideal DCG:
    # If there are N relevant documents, the ideal ranking
    # puts all relevant documents at the top.
    num_relevant = min(len(set(gold_doc_ids)), k)

    idcg = sum(
        1.0 / math.log2(rank + 1)
        for rank in range(1, num_relevant + 1)
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg


## **Evaluate Retreival Metrics**

In [14]:
rows = []

for query_idx, query in enumerate(data):

    gold_doc_ids = query["gold_doc_ids"]

    for method_name, method_key in METHODS.items():

        ranked_results = query["runs"][method_key]

        row = {
            "query_idx": query_idx,
            "method": method_name,
        }

        for k in K_VALUES:

            row[f"Recall@{k}"] = recall_at_k(
                ranked_results,
                gold_doc_ids,
                k
            )

            row[f"MRR@{k}"] = reciprocal_rank_at_k(
                ranked_results,
                gold_doc_ids,
                k
            )

            row[f"NDCG@{k}"] = ndcg_at_k(
                ranked_results,
                gold_doc_ids,
                k
            )

        rows.append(row)


per_query_df = pd.DataFrame(rows)

print("\nPer-query results:")
print(per_query_df.head())


Per-query results:
   query_idx                              method  Recall@3  MRR@3  NDCG@3  \
0          0                                BM25       0.0    0.0     0.0   
1          0              BM25 + query expansion       0.0    0.0     0.0   
2          0                    BM25 + reranking       0.0    0.0     0.0   
3          0  BM25 + query expansion + reranking       0.0    0.0     0.0   
4          1                                BM25       1.0    1.0     1.0   

   Recall@5  MRR@5  NDCG@5  Recall@7  MRR@7  NDCG@7  Recall@10  MRR@10  \
0       0.0    0.0     0.0       0.0    0.0     0.0        0.0     0.0   
1       0.0    0.0     0.0       0.0    0.0     0.0        0.0     0.0   
2       0.0    0.0     0.0       0.0    0.0     0.0        0.0     0.0   
3       0.0    0.0     0.0       0.0    0.0     0.0        0.0     0.0   
4       1.0    1.0     1.0       1.0    1.0     1.0        1.0     1.0   

   NDCG@10  Recall@20  MRR@20  NDCG@20  Recall@50  MRR@50  NDCG@50  
0  

In [15]:
metric_columns = []

for metric in ["Recall", "MRR", "NDCG"]:
    for k in K_VALUES:
        metric_columns.append(f"{metric}@{k}")


average_df = (
    per_query_df
    .groupby("method")[metric_columns]
    .mean()
    .reset_index()
)

## **Display Retrieval Results**

In [16]:
recall_table = average_df[
    ["method"] + [f"Recall@{k}" for k in K_VALUES]
].copy()

mrr_table = average_df[
    ["method"] + [f"MRR@{k}" for k in K_VALUES]
].copy()

ndcg_table = average_df[
    ["method"] + [f"NDCG@{k}" for k in K_VALUES]
].copy()


# Rename Method column exactly as requested
recall_table = recall_table.rename(columns={"method": "Method"})
mrr_table = mrr_table.rename(columns={"method": "Method"})
ndcg_table = ndcg_table.rename(columns={"method": "Method"})


# Round for presentation
recall_table = recall_table.round(4)
mrr_table = mrr_table.round(4)
ndcg_table = ndcg_table.round(4)

In [17]:
print("\n" + "=" * 80)
print("RECALL")
print("=" * 80)
print(recall_table.to_string(index=False))

print("\n" + "=" * 80)
print("MRR")
print("=" * 80)
print(mrr_table.to_string(index=False))

print("\n" + "=" * 80)
print("NDCG")
print("=" * 80)
print(ndcg_table.to_string(index=False))


RECALL
                            Method  Recall@3  Recall@5  Recall@7  Recall@10  Recall@20  Recall@50
                              BM25    0.6861    0.7469    0.7812     0.8128     0.8517     0.8841
            BM25 + query expansion    0.6503    0.7278    0.7636     0.8136     0.8432     0.8767
BM25 + query expansion + reranking    0.6689    0.7359    0.7717     0.8049     0.8560     0.8767
                  BM25 + reranking    0.7278    0.7697    0.7988     0.8387     0.8691     0.8841

MRR
                            Method  MRR@3  MRR@5  MRR@7  MRR@10  MRR@20  MRR@50
                              BM25 0.6111 0.6253 0.6308  0.6338  0.6364  0.6372
            BM25 + query expansion 0.5956 0.6119 0.6172  0.6226  0.6245  0.6256
BM25 + query expansion + reranking 0.6061 0.6194 0.6251  0.6290  0.6326  0.6334
                  BM25 + reranking 0.6556 0.6646 0.6693  0.6741  0.6759  0.6763

NDCG
                            Method  NDCG@3  NDCG@5  NDCG@7  NDCG@10  NDCG@20  NDCG@50
     

## **Save**

In [18]:
recall_table.to_csv("recall_results.csv", index=False)
mrr_table.to_csv("mrr_results.csv", index=False)
ndcg_table.to_csv("ndcg_results.csv", index=False)

print("\nSaved:")
print("  recall_results.csv")
print("  mrr_results.csv")
print("  ndcg_results.csv")


Saved:
  recall_results.csv
  mrr_results.csv
  ndcg_results.csv
